# Message Intelligence System


> The **Message Intelligence System** project focuses on building a classification system that automatically identifies whether incoming digital messages are **Spam** or **Legitimate**. The project combines probability theory with distance-based, margin-based, and probabilistic classifiers — K-Nearest Neighbors, Support Vector Machine, and Naive Bayes — to compare how different mathematical assumptions affect real classification performance.

> The final goal is to evaluate all three models, understand their individual strengths, and recommend the best approach for real-world deployment in a communication security setting.



## Problem Statement

> You are working as a Data Scientist for a communication security company. The company wants to build a Message Intelligence System that can automatically classify user messages as:

- **0 → Legitimate Message**
- **1 → Spam Message**

> The dataset contains message-related features extracted from text (numerical summaries) and user behavior signals. Some features are highly correlated, while others follow probabilistic patterns. Your task is to build and compare multiple classification models and explain their performance using probability concepts.


# Part A: Probability & Conceptual Foundation (Theory)


## 1. What is Conditional Probability?

Conditional Probability is the probability of an event occurring given that another event has already occurred. It is written as P(A|B), meaning the probability of A happening given that B is true. It forms the mathematical foundation for most probabilistic classifiers.

---

## 2. Explain Bayes' Theorem and its importance in classification problems.

Bayes' Theorem calculates the probability of a class given observed features, using prior knowledge and likelihood: P(Class | Features) = [P(Features | Class) × P(Class)] / P(Features). It is important in classification because it lets a model update its belief about the class once evidence (features) is known, forming the basis of the Naive Bayes Classifier.

---

## 3. What assumptions does the Naive Bayes Classifier make?

Naive Bayes assumes that all features are **conditionally independent** of each other given the class label. It also assumes each feature contributes independently to the final probability calculation. This assumption is often unrealistic in real data, but the model still performs well in practice.

---

## 4. Explain the working principle of KNN and SVM.

**K-Nearest Neighbors (KNN):** KNN classifies a new data point by looking at the 'K' closest data points in the training set (based on a distance metric) and assigning the majority class among those neighbours. It does not build an explicit model — it stores the training data and decides at prediction time.

**Support Vector Machine (SVM):** SVM finds the optimal hyperplane that separates two classes with the **maximum margin**. Data points closest to this boundary are called support vectors. SVM can use kernel functions (linear, RBF, polynomial) to separate data that is not linearly separable in its original space.

---

## 5. Compare distance-based, probabilistic, and margin-based classifiers.

**Distance-based (KNN):** Classifies based on proximity to neighbouring points. Simple, but sensitive to feature scale and noisy data.\
**Probabilistic (Naive Bayes):** Classifies based on calculated class probabilities using Bayes' Theorem. Fast and works well with limited data, but assumes feature independence.\
**Margin-based (SVM):** Classifies by finding the boundary with the maximum margin between classes. Performs well on high-dimensional data, but can be slower to train on large datasets.


# Part B: Dataset Understanding & Preparation


## Dataset Description

| Feature | Description |
|---|---|
| message_id | Unique message ID (dropped before modelling) |
| message_text | Original message text (dropped — numeric features already extracted) |
| message_length | Number of characters in the message |
| word_count | Number of words in the message |
| num_urls | Number of URLs present in the message |
| num_digits | Number of digits present in the message |
| num_special_chars | Number of special characters (!, $, % etc.) |
| spam_keyword_score | Score based on presence of spam-related keywords |
| legit_keyword_score | Score based on presence of legitimate/business keywords |
| sender_activity_score | Behavioral score of the sender's activity |
| sender_account_age_days | Age of the sender's account in days |
| messages_sent_last_24h | Number of messages sent by sender in the last 24 hours |
| timestamp | Date and time the message was sent (used to derive hour/day features) |
| hour_of_day | Hour of the day the message was sent (0–23) |
| day_of_week | Day of the week the message was sent (0–6) |
| spam_label | **Target**: 0 = Legitimate, 1 = Spam |


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel("Message_Intelligence_Dataset.xlsx")

In [3]:
display(df)

,message_id,message_text,message_length,word_count,num_urls,num_digits,num_special_chars,spam_keyword_score,legit_keyword_score,sender_activity_score,sender_account_age_days,messages_sent_last_24h,timestamp,hour_of_day,day_of_week,spam_label
0,900001,Please find the attached invoice for the updat...,99,11,1,4,0,0,1,56.6,500.0,6.0,2025-11-24 02:00:00,2,0,0
1,900002,Let's catch up tomorrow regarding the timeline...,73,12,0,0,0,0,0,16.6,207.0,0.0,2025-12-17 21:00:00,21,2,0
2,900003,Can you send the report by end of day? next Mo...,67,13,0,0,0,0,1,25.7,418.0,6.0,2025-11-15 13:00:00,13,5,0
3,900004,Can you send the report by end of day? 10:30 A...,64,13,0,4,0,0,1,48.8,276.0,5.0,2025-12-17 23:00:00,23,2,0
4,900005,Could you review the document and share feedba...,84,14,0,0,0,0,1,33.0,683.0,7.0,2025-11-29 11:00:00,11,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5195,905196,Thanks for your help on the project update. th...,73,12,0,0,0,0,1,23.1,262.0,5.0,2025-10-28 13:00:00,13,1,0
5196,905197,Let's catch up tomorrow regarding the timeline...,108,12,1,4,0,0,0,62.4,398.0,6.0,2025-11-25 14:00:00,14,1,0
5197,905198,Here are the notes from today's discussion. th...,80,12,0,0,0,0,1,96.6,457.0,5.0,2025-10-29 14:00:00,14,2,0
5198,905199,Act fast! Your account will be suspended unles...,94,14,1,4,2,1,0,70.1,9.0,11.0,2025-11-20 23:00:00,23,3,1


In [4]:
print("Dataset Shape :", df.shape)

Dataset Shape : (5200, 16)


In [5]:
print("Column Names :")
print(df.columns.tolist())

Column Names :
['message_id', 'message_text', 'message_length', 'word_count', 'num_urls', 'num_digits', 'num_special_chars', 'spam_keyword_score', 'legit_keyword_score', 'sender_activity_score', 'sender_account_age_days', 'messages_sent_last_24h', 'timestamp', 'hour_of_day', 'day_of_week', 'spam_label']


In [6]:
print("Data Types :")
print(df.dtypes)

Data Types :
message_id                          int64
message_text                       object
message_length                      int64
word_count                          int64
num_urls                            int64
num_digits                          int64
num_special_chars                   int64
spam_keyword_score                  int64
legit_keyword_score                 int64
sender_activity_score             float64
sender_account_age_days           float64
messages_sent_last_24h            float64
timestamp                  datetime64[ns]
hour_of_day                         int64
day_of_week                         int64
spam_label                          int64
dtype: object


In [7]:
print("\nTarget Class Distribution :")
print(df["spam_label"].value_counts())
print("\nClass Percentage :")
print(df["spam_label"].value_counts(normalize=True).round(3))


Target Class Distribution :
spam_label
0    4227
1     973
Name: count, dtype: int64

Class Percentage :
spam_label
0    0.813
1    0.187
Name: proportion, dtype: float64


### Interpretation

> The dataset has 5,200 messages with 81% Legitimate and 19% Spam — a moderate class imbalance.\
> The dataset includes both text-derived numerical features and sender behavior signals.\
> `message_text` and `message_id` are identifiers/raw text and are not used directly as model features.


In [8]:
print("Missing Values per Column :")
print(df.isnull().sum())

Missing Values per Column :
message_id                   0
message_text                 0
message_length               0
word_count                   0
num_urls                     0
num_digits                   0
num_special_chars            0
spam_keyword_score           0
legit_keyword_score          0
sender_activity_score      106
sender_account_age_days    113
messages_sent_last_24h     162
timestamp                    0
hour_of_day                  0
day_of_week                  0
spam_label                   0
dtype: int64


### Feature Preparation — Removing Non-Predictive Columns


In [9]:
# message_id is a unique identifier — no predictive value
# message_text is raw text — numeric features (length, keyword scores, etc.) already extracted from it
# timestamp is already broken into hour_of_day and day_of_week
df_ml = df.drop(['message_id', 'message_text', 'timestamp'], axis=1)

print("Columns used for modelling :")
print(df_ml.columns.tolist())

Columns used for modelling :
['message_length', 'word_count', 'num_urls', 'num_digits', 'num_special_chars', 'spam_keyword_score', 'legit_keyword_score', 'sender_activity_score', 'sender_account_age_days', 'messages_sent_last_24h', 'hour_of_day', 'day_of_week', 'spam_label']


### 6. Identify Input Features and Target Variable


In [10]:
X = df_ml.drop('spam_label', axis=1)
y = df_ml['spam_label']

print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

Feature Shape : (5200, 12)
Target Shape  : (5200,)


### Handle Missing Values


In [11]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

print("Missing values after imputation :")
print(X_imputed.isnull().sum())

Missing values after imputation :
message_length             0
word_count                 0
num_urls                   0
num_digits                 0
num_special_chars          0
spam_keyword_score         0
legit_keyword_score        0
sender_activity_score      0
sender_account_age_days    0
messages_sent_last_24h     0
hour_of_day                0
day_of_week                0
dtype: int64


### 8. Split the Dataset into Training and Testing Sets


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Set Shape :", X_train.shape)
print("Testing Set Shape  :", X_test.shape)

Training Set Shape : (4160, 12)
Testing Set Shape  : (1040, 12)


### 7. Apply Feature Scaling


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

print("Feature scaling applied successfully !")

Feature scaling applied successfully !


### Interpretation

> Scaling is essential here because KNN and SVM rely on distance and margin calculations. Features like `message_length` and `sender_account_age_days` have very different ranges, and without scaling they would dominate the distance calculation unfairly.


# Part C: Baseline Model — K-Nearest Neighbors


In [14]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

### 9. Implement K-Nearest Neighbors (KNN) Classifier


In [15]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [16]:
y_pred_knn = knn_model.predict(X_test_scaled)

In [17]:
print("KNN (k=5) Results :")
print("Accuracy  :", accuracy_score(y_test, y_pred_knn))
print("Precision :", precision_score(y_test, y_pred_knn))
print("Recall    :", recall_score(y_test, y_pred_knn))
print("F1-Score  :", f1_score(y_test, y_pred_knn))

KNN (k=5) Results :
Accuracy  : 1.0
Precision : 1.0
Recall    : 1.0
F1-Score  : 1.0


### 10. Experiment with Different Values of K


In [18]:
k_values = [1, 3, 5, 7, 9, 11, 15]
k_results = []

for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_scaled, y_train)
    pred_k = knn_k.predict(X_test_scaled)
    k_results.append({
        'K': k,
        'Accuracy': accuracy_score(y_test, pred_k),
        'Precision': precision_score(y_test, pred_k),
        'Recall': recall_score(y_test, pred_k),
        'F1-Score': f1_score(y_test, pred_k)
    })

k_results_df = pd.DataFrame(k_results)
print(k_results_df)

    K  Accuracy  Precision    Recall  F1-Score
0   1  1.000000        1.0  1.000000  1.000000
1   3  1.000000        1.0  1.000000  1.000000
2   5  1.000000        1.0  1.000000  1.000000
3   7  0.999038        1.0  0.994872  0.997429
4   9  0.999038        1.0  0.994872  0.997429
5  11  0.999038        1.0  0.994872  0.997429
6  15  0.999038        1.0  0.994872  0.997429


### Interpretation

> Smaller values of K (like 1 or 3) make the model sensitive to noise in individual data points.\
> Larger values of K make the decision boundary smoother but may blur the distinction between classes.\
> K=5 gives a good balance between accuracy and stability for this dataset.


### 11. Analyze How Distance Metrics Affect Predictions


In [39]:
distance_metrics = ['euclidean', 'manhattan', 'minkowski']
metric_results = []

for metric in distance_metrics:
    knn_m = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn_m.fit(X_train_scaled, y_train)
    pred_m = knn_m.predict(X_test_scaled)
    metric_results.append({
        'Distance Metric': metric,
        'Accuracy': accuracy_score(y_test, pred_m),
        'F1-Score': f1_score(y_test, pred_m)
    })

metric_results_df = pd.DataFrame(metric_results)
print(metric_results_df)

  Distance Metric  Accuracy  F1-Score
0       euclidean       1.0       1.0
1       manhattan       1.0       1.0
2       minkowski       1.0       1.0


### Interpretation

> **Euclidean distance** measures straight-line distance and works well when features are scaled.\
> **Manhattan distance** sums absolute differences and is more robust to outliers.\
> **Minkowski distance** is a generalized form that becomes Euclidean or Manhattan depending on its parameter.\
> All three metrics give similar results here because the features are well-scaled and the classes are well separated.


### 12. Identify Cases Where KNN Misclassifies Messages


In [20]:
knn_final = KNeighborsClassifier(n_neighbors=5)
knn_final.fit(X_train_scaled, y_train)
y_pred_final = knn_final.predict(X_test_scaled)

misclassified_mask = (y_pred_final != y_test.values)
misclassified_indices = X_test.index[misclassified_mask]

print("Number of Misclassified Messages :", len(misclassified_indices))

if len(misclassified_indices) > 0:
    sample_idx = misclassified_indices[0]
    print("\nExample Misclassified Message :")
    print(df.loc[sample_idx, ['message_text', 'spam_label']])
    print("Predicted as :", y_pred_final[list(X_test.index).index(sample_idx)])
else:
    print("No misclassifications found on the test set.")

Number of Misclassified Messages : 0
No misclassifications found on the test set.


### Interpretation

> Misclassifications usually happen when a message has mixed signals — for example, a spam message with very few special characters or a low spam-keyword score, making it numerically resemble a legitimate message.\
> KNN struggles most with borderline cases that sit close to the decision boundary between classes.


# Part D: Support Vector Machine Classifier


In [21]:
from sklearn.svm import SVC

### 13. Implement SVM with Linear Kernel


In [22]:
svm_linear = SVC(kernel='linear', probability=True, random_state=42)
svm_linear.fit(X_train_scaled, y_train)
y_pred_svm_linear = svm_linear.predict(X_test_scaled)

In [23]:
print("SVM (Linear Kernel) Results :")
print("Accuracy  :", accuracy_score(y_test, y_pred_svm_linear))
print("Precision :", precision_score(y_test, y_pred_svm_linear))
print("Recall    :", recall_score(y_test, y_pred_svm_linear))
print("F1-Score  :", f1_score(y_test, y_pred_svm_linear))

SVM (Linear Kernel) Results :
Accuracy  : 1.0
Precision : 1.0
Recall    : 1.0
F1-Score  : 1.0


### 13. Implement SVM with RBF and Polynomial Kernel


In [24]:
svm_rbf = SVC(kernel='rbf', probability=True, random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_svm_rbf = svm_rbf.predict(X_test_scaled)

svm_poly = SVC(kernel='poly', degree=3, probability=True, random_state=42)
svm_poly.fit(X_train_scaled, y_train)
y_pred_svm_poly = svm_poly.predict(X_test_scaled)

In [25]:
svm_kernel_results = pd.DataFrame({
    'Kernel': ['Linear', 'RBF', 'Polynomial'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_svm_linear),
        accuracy_score(y_test, y_pred_svm_rbf),
        accuracy_score(y_test, y_pred_svm_poly)
    ],
    'Precision': [
        precision_score(y_test, y_pred_svm_linear),
        precision_score(y_test, y_pred_svm_rbf),
        precision_score(y_test, y_pred_svm_poly)
    ],
    'Recall': [
        recall_score(y_test, y_pred_svm_linear),
        recall_score(y_test, y_pred_svm_rbf),
        recall_score(y_test, y_pred_svm_poly)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_svm_linear),
        f1_score(y_test, y_pred_svm_rbf),
        f1_score(y_test, y_pred_svm_poly)
    ]
})

print(svm_kernel_results)

       Kernel  Accuracy  Precision  Recall  F1-Score
0      Linear       1.0        1.0     1.0       1.0
1         RBF       1.0        1.0     1.0       1.0
2  Polynomial       1.0        1.0     1.0       1.0


### 14. Analyze Margin Separation and Support Vectors


In [26]:
print("Number of Support Vectors per Class (Linear Kernel) :", svm_linear.n_support_)
print("Total Support Vectors :", sum(svm_linear.n_support_))
print("Total Training Samples :", len(X_train_scaled))
print("Support Vector Percentage :", round(sum(svm_linear.n_support_) / len(X_train_scaled) * 100, 2), "%")

Number of Support Vectors per Class (Linear Kernel) : [5 5]
Total Support Vectors : 10
Total Training Samples : 4160
Support Vector Percentage : 0.24 %


### Interpretation

> Support vectors are the data points closest to the decision boundary — they are the only points that actually influence where the margin is placed.\
> A small number of support vectors relative to the training set size means the two classes (Spam and Legitimate) are well separated, and the margin is wide and confident.


### 15. Compare SVM Performance with KNN


In [27]:
svm_vs_knn = pd.DataFrame({
    'Model': ['KNN (k=5)', 'SVM (Linear)', 'SVM (RBF)', 'SVM (Polynomial)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_svm_linear),
        accuracy_score(y_test, y_pred_svm_rbf),
        accuracy_score(y_test, y_pred_svm_poly)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_svm_linear),
        f1_score(y_test, y_pred_svm_rbf),
        f1_score(y_test, y_pred_svm_poly)
    ]
})

print(svm_vs_knn)

              Model  Accuracy  F1-Score
0         KNN (k=5)       1.0       1.0
1      SVM (Linear)       1.0       1.0
2         SVM (RBF)       1.0       1.0
3  SVM (Polynomial)       1.0       1.0


### Interpretation

> Both KNN and SVM perform very well on this dataset because the engineered features (special characters, keyword scores, sender behaviour) separate Spam and Legitimate messages clearly.\
> SVM tends to generalize slightly better on unseen data because it optimizes for the widest margin, rather than relying purely on local neighbours like KNN.


# Part E: Naive Bayes Classifier & Probability


### 16. Implement Naive Bayes Classifier


In [28]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
y_pred_nb = nb_model.predict(X_test_scaled)

In [29]:
print("Naive Bayes Results :")
print("Accuracy  :", accuracy_score(y_test, y_pred_nb))
print("Precision :", precision_score(y_test, y_pred_nb))
print("Recall    :", recall_score(y_test, y_pred_nb))
print("F1-Score  :", f1_score(y_test, y_pred_nb))

print("\nConfusion Matrix :")
print(confusion_matrix(y_test, y_pred_nb))

Naive Bayes Results :
Accuracy  : 1.0
Precision : 1.0
Recall    : 1.0
F1-Score  : 1.0

Confusion Matrix :
[[845   0]
 [  0 195]]


### 17. Manually Compute Conditional Probabilities for Sample Messages


In [30]:
# Using 'has_url' (whether a message contains at least one URL) as the evidence
df['has_url'] = (df['num_urls'] >= 1).astype(int)

# P(Spam) and P(Legitimate) — prior probabilities
p_spam = df['spam_label'].mean()
p_legit = 1 - p_spam

print("P(Spam)       :", round(p_spam, 4))
print("P(Legitimate) :", round(p_legit, 4))

P(Spam)       : 0.1871
P(Legitimate) : 0.8129


In [31]:
# P(HasURL | Spam) and P(HasURL | Legitimate) — likelihoods
p_url_given_spam  = df[df['spam_label'] == 1]['has_url'].mean()
p_url_given_legit = df[df['spam_label'] == 0]['has_url'].mean()

print("P(HasURL | Spam)       :", round(p_url_given_spam, 4))
print("P(HasURL | Legitimate) :", round(p_url_given_legit, 4))

P(HasURL | Spam)       : 0.739
P(HasURL | Legitimate) : 0.084


In [32]:
# P(HasURL) — overall (evidence) probability
p_url = df['has_url'].mean()
print("P(HasURL) :", round(p_url, 4))

P(HasURL) : 0.2065


### 18. Apply Bayes' Theorem to Compute Class Probability


In [33]:
# Bayes' Theorem:
# P(Spam | HasURL) = [ P(HasURL | Spam) * P(Spam) ] / P(HasURL)

p_spam_given_url = (p_url_given_spam * p_spam) / p_url

print("P(Spam | HasURL) =", round(p_spam_given_url, 4))
print("\nInterpretation: If a message contains a URL, there is about",
      round(p_spam_given_url * 100, 1), "% probability it is Spam.")

P(Spam | HasURL) = 0.6695

Interpretation: If a message contains a URL, there is about 66.9 % probability it is Spam.


### Interpretation

> Before seeing any evidence, the prior probability of Spam is only about 19%.\
> Once we observe that a message contains a URL, the probability of Spam jumps to roughly 67%.\
> This shows exactly how Bayes' Theorem updates our belief about the class once new evidence is introduced — this is the same core idea the Naive Bayes Classifier uses internally, but applied across all features at once.


### 19. Compare Theoretical Probability with Model Predictions


In [42]:
# Get model's predicted probability for messages that contain a URL
url_test_mask = (X_test['num_urls'] >= 1)
model_probs_for_url_messages = nb_model.predict_proba(X_test_scaled[url_test_mask.values])[:, 1]

print("Manual Bayes Calculation : P(Spam | HasURL) =", round(p_spam_given_url, 4))
print("Naive Bayes Model Average Predicted P(Spam | HasURL) =", round(model_probs_for_url_messages.mean(), 4))

Manual Bayes Calculation : P(Spam | HasURL) = 0.6695
Naive Bayes Model Average Predicted P(Spam | HasURL) = 0.6222


### Interpretation

> The manually calculated probability using a single feature (HasURL) is close to the Naive Bayes model's average predicted probability for the same group of messages.\
> The small difference exists because the model combines **all features together** (keyword scores, special characters, sender behaviour, etc.), not just the URL feature alone, giving a more complete probability estimate.


# Part F: Model Comparison & Evaluation


### 20. Evaluate All Models Using Classification Metrics


In [35]:
all_models_results = pd.DataFrame({
    'Model': ['KNN', 'SVM (Linear)', 'SVM (RBF)', 'SVM (Polynomial)', 'Naive Bayes'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_svm_linear),
        accuracy_score(y_test, y_pred_svm_rbf),
        accuracy_score(y_test, y_pred_svm_poly),
        accuracy_score(y_test, y_pred_nb)
    ],
    'Precision': [
        precision_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_svm_linear),
        precision_score(y_test, y_pred_svm_rbf),
        precision_score(y_test, y_pred_svm_poly),
        precision_score(y_test, y_pred_nb)
    ],
    'Recall': [
        recall_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_svm_linear),
        recall_score(y_test, y_pred_svm_rbf),
        recall_score(y_test, y_pred_svm_poly),
        recall_score(y_test, y_pred_nb)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_svm_linear),
        f1_score(y_test, y_pred_svm_rbf),
        f1_score(y_test, y_pred_svm_poly),
        f1_score(y_test, y_pred_nb)
    ]
})

print(all_models_results)

              Model  Accuracy  Precision  Recall  F1-Score
0               KNN       1.0        1.0     1.0       1.0
1      SVM (Linear)       1.0        1.0     1.0       1.0
2         SVM (RBF)       1.0        1.0     1.0       1.0
3  SVM (Polynomial)       1.0        1.0     1.0       1.0
4       Naive Bayes       1.0        1.0     1.0       1.0


### 21. Compare KNN vs SVM vs Naive Bayes


In [36]:
comparison_summary = pd.DataFrame({
    'Aspect': ['Approach', 'Speed', 'Interpretability', 'Sensitive to Scaling', 'Best For'],
    'KNN': ['Distance-based', 'Slow on large data', 'Medium', 'Yes', 'Simple, well-separated data'],
    'SVM': ['Margin-based', 'Moderate', 'Low (especially RBF/Poly)', 'Yes', 'High-dimensional, complex boundaries'],
    'Naive Bayes': ['Probability-based', 'Very Fast', 'High', 'No', 'Quick baseline, text-style data']
})

print(comparison_summary.to_string(index=False))

              Aspect                         KNN                                  SVM                     Naive Bayes
            Approach              Distance-based                         Margin-based               Probability-based
               Speed          Slow on large data                             Moderate                       Very Fast
    Interpretability                      Medium            Low (especially RBF/Poly)                            High
Sensitive to Scaling                         Yes                                  Yes                              No
            Best For Simple, well-separated data High-dimensional, complex boundaries Quick baseline, text-style data


### Interpretation

> All three models perform strongly on this dataset because the engineered features create a clear separation between Spam and Legitimate messages.\
> KNN and SVM rely on geometric closeness/margins, while Naive Bayes relies purely on probability — yet all three reach similar conclusions, which confirms the features themselves are highly informative.


### 22. Identify Which Model Performs Best for High Precision / High Recall


In [37]:
best_precision_model = all_models_results.sort_values('Precision', ascending=False).iloc[0]
best_recall_model = all_models_results.sort_values('Recall', ascending=False).iloc[0]

print("Best Model for High Precision :")
print(best_precision_model)

print("\nBest Model for High Recall :")
print(best_recall_model)

Best Model for High Precision :
Model        KNN
Accuracy     1.0
Precision    1.0
Recall       1.0
F1-Score     1.0
Name: 0, dtype: object

Best Model for High Recall :
Model        KNN
Accuracy     1.0
Precision    1.0
Recall       1.0
F1-Score     1.0
Name: 0, dtype: object


### Interpretation

> **High Precision** matters when we want to avoid wrongly flagging legitimate messages as spam (reducing false alarms for genuine users).\
> **High Recall** matters when we want to catch as many spam messages as possible, even if a few legitimate messages get flagged by mistake.\
> For a communication security system, catching spam (high Recall) is usually the higher priority, since missed spam can lead to phishing or fraud reaching real users.


# Part G: Final Analysis & Reporting


#### A. Strengths and Weaknesses of Each Classifier


### K-Nearest Neighbors (KNN)

### Advantages

* Easy to understand and implement.
* No training phase is required.
* Works directly with stored data.

### Disadvantages

* Slow for large datasets because it compares with all training samples.
* Sensitive to irrelevant features.
* Performance depends on the choice of K value.

---

## Support Vector Machine (SVM)

### Advantages

* Finds the optimal decision boundary for classification.
* Handles non-linear data using kernel functions.
* Provides good generalization on unseen data.

### Disadvantages

* Difficult to interpret compared to simpler models.
* Training can be slow on large datasets.
* Requires careful selection of kernel and parameters.

---

## Naive Bayes

### Advantages

* Fast training and prediction.
* Simple and computationally efficient.
* Produces probabilistic predictions.

### Disadvantages

* Assumes all features are independent.
* Performance may decrease when features are highly correlated.
* May not capture complex relationships between features.


#### B. Impact of Probability Assumptions in Naive Bayes


- Naive Bayes assumes all features are independent given the class label — in this dataset, features like `spam_keyword_score` and `num_special_chars` are likely correlated with each other.
- Despite this unrealistic assumption, Naive Bayes still performed competitively, showing that the 'naive' independence assumption does not always hurt real-world performance.
- This is a well-known property of Naive Bayes — it can work well even when its core assumption is technically violated.


#### C. Trade-offs Between Interpretability and Performance


- **Naive Bayes** is the most interpretable — every prediction can be explained using clear probabilities.
- **KNN** is moderately interpretable — predictions can be traced back to specific neighbouring examples.
- **SVM** (especially with RBF/Polynomial kernels) is the least interpretable — the decision boundary is harder to explain to a non-technical stakeholder.
- In regulated industries like banking or communication security, the choice between interpretability and raw performance is an important business decision, not just a technical one.


#### D. Business Recommendation for Real-World Deployment


- For a production communication security system, **SVM (Linear Kernel)**  as the primary model — it offers strong, stable performance and is faster to deploy than non-linear kernels.
- **Naive Bayes** is a lightweight, fast secondary filter that can run instantly at scale (e.g., on millions of messages per day) before a heavier model is applied.
- **KNN** is least suitable for large-scale deployment due to slower prediction times as the dataset grows, but remains useful for smaller-scale testing and prototyping.
- A combination approach — Naive Bayes for fast initial filtering, SVM for final confirmation — balances speed and accuracy in a real system.


## Final Conclusion


- The dataset contained 5,200 messages with 81% Legitimate and 19% Spam — a moderately imbalanced classification problem.
- Engineered features such as `num_special_chars`, `spam_keyword_score`, and `legit_keyword_score` proved highly informative in separating Spam from Legitimate messages.
- KNN, SVM, and Naive Bayes all achieved strong performance, confirming the dataset's features carry a clear, learnable signal.
- The manual Bayes' Theorem calculation closely matched the Naive Bayes model's own predicted probabilities, confirming the model is working as mathematically expected.
- SVM is selected as the primary recommended model for deployment due to its strong generalization through margin maximization, with Naive Bayes as a fast complementary filter.
- This Message Intelligence System is suitable for deployment as an automated spam-detection layer in a communication security platform.
